# Congestion, Convergence, and Counter-Intuitive Flows: A Toolkit

Iacopo Bertelli's [Counterintuitive Flows in 2023 (Part 2 of 3)](https://www.linkedin.com/pulse/counterintuitive-flows-2023-part-2-3-iacopo-bertelli-baonf) manually dug through a year of ENTSO-E data to find three specific moments: the hour European day-ahead prices converged almost everywhere, the hour the most bidding-zone borders were simultaneously congested, and the hour the most borders had power flowing the "wrong" way relative to the price signal.

That's exactly the kind of one-off, hand-rolled analysis that's hard for anyone else to check or repeat. This notebook builds three small, reusable functions on top of the Voltpath API &mdash; one each for congestion, counter-intuitive flow, and price convergence &mdash; that take any date range and return the same kind of answer. We use them to reproduce Iacopo's three 2023 moments, then point the same functions at real structural shifts in the European grid from 2024&ndash;2026 to see whether they show up in the data.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

**Before you run this:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key, then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY` (or leave `.env` empty and you'll be prompted).

In [ ]:
import getpass
import os

import pandas as pd
import plotly.express as px
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://api.voltpath.ai"
API_KEY = os.environ.get("VOLTPATH_API_KEY") or getpass.getpass("Voltpath API key (from voltpath.ai): ")
HEADERS = {"x-api-key": API_KEY} if API_KEY else {}

# Same flow-type colours as the Voltpath dashboard (apps/web/src/hooks/colors.ts)
FLOW_COLORS = {
    "Convergence": "#08415C",
    "Congestion": "#CC2936",
    "Counter-intuitive": "#6B818C",
    "Outage": "#F1BF98",
}

## Fetching flow data

`GET /api/v1/flows` joins physical flow with day-ahead prices on both sides of a border and classifies each observation into `Convergence`, `Congestion`, `Counter-intuitive`, or `Outage` &mdash; see [Flow Types](https://docs.voltpath.ai/concepts/flow-types). The API has no limit on how far back `datetime_from`/`datetime_to` can reach; it just downgrades resolution automatically as the range widens (native for &le;2 days, hourly for &le;14, daily beyond) to keep responses a manageable size. Pass `resolution="native"` explicitly to keep 15-minute granularity on short windows.

`fetch_flows` wraps that call and adds a few columns every analysis below needs: a human-readable `border` label, each observation's duration (`duration_h`, inferred from the gap between consecutive timestamps since different borders report at different cadences), the traded volume in MWh, and the implied economic value of the flow (`price_delta x net_flow_mw x duration_h`, signed).

In [ ]:
def fetch_flows(date_from: str, date_to: str, resolution: str | None = None) -> pd.DataFrame:
    params = {"datetime_from": date_from, "datetime_to": date_to}
    if resolution:
        params["resolution"] = resolution
    resp = requests.get(f"{BASE_URL}/api/v1/flows", headers=HEADERS, params=params, timeout=60)
    resp.raise_for_status()

    df = pd.DataFrame(resp.json()["data"])
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["border"] = df["bidding_zone_from"] + " -> " + df["bidding_zone_to"]

    diffs_h = df.sort_values("datetime").groupby("border")["datetime"].diff().dt.total_seconds() / 3600
    median_cadence = diffs_h.groupby(df["border"]).transform("median")
    df["duration_h"] = diffs_h.fillna(median_cadence)

    df["volume_mwh"] = df["net_flow_mw"].abs() * df["duration_h"]
    df["value_eur"] = df["price_delta"] * df["net_flow_mw"] * df["duration_h"]
    return df

## Reusable analysis functions

Three questions, three small functions, all built on the same `fetch_flows` output:

- `hourly_border_counts(df, flow_type)` &mdash; how many distinct borders are in a given state at each timestamp. Its `.idxmax()` is exactly Iacopo's "hour with the most X" headline number; the full series is the trend around it.
- `flow_type_summary(df, flow_type)` &mdash; per border, total volume traded (MWh) and the implied welfare value (`|price_delta x net_flow_mw x duration_h|`, summed). For `Congestion`, that's the unrealised gain from trade the bottleneck is blocking. For `Counter-intuitive`, it's the per-leg loss or gain from flowing against (or with) the price gradient on that specific border.
- `convergence_summary(df)` &mdash; per border, volume traded while converged and the share of the window spent converged. Convergence has no welfare-gap story of its own (`price_delta &asymp; 0` there by definition), so volume and time-share are the meaningful stats instead.

In [ ]:
def hourly_border_counts(df: pd.DataFrame, flow_type: str) -> pd.DataFrame:
    subset = df[df["flow_type"] == flow_type]
    return subset.groupby("datetime")["border"].nunique().rename("n_borders").reset_index()


def peak(counts: pd.DataFrame) -> pd.Series:
    return counts.loc[counts["n_borders"].idxmax()]


def flow_type_summary(df: pd.DataFrame, flow_type: str) -> pd.DataFrame:
    subset = df[df["flow_type"] == flow_type]
    return (
        subset.groupby("border")
        .agg(volume_mwh=("volume_mwh", "sum"), implied_value_eur=("value_eur", lambda s: s.abs().sum()))
        .sort_values("implied_value_eur", ascending=False)
        .reset_index()
    )


def convergence_summary(df: pd.DataFrame) -> pd.DataFrame:
    total_hours = df.groupby("border")["duration_h"].sum()
    conv = df[df["flow_type"] == "Convergence"]
    out = conv.groupby("border").agg(volume_mwh=("volume_mwh", "sum"), converged_hours=("duration_h", "sum"))
    out["share_converged"] = out["converged_hours"] / total_hours.reindex(out.index)
    return out.sort_values("volume_mwh", ascending=False).reset_index()

## Part 1: reproducing Iacopo's three 2023 moments

[Counterintuitive Flows in 2023, Part 2 of 3](https://www.linkedin.com/pulse/counterintuitive-flows-2023-part-2-3-iacopo-bertelli-baonf) called out three specific hours. We don't chase his exact border counts &mdash; a manual, one-off analysis from a year ago isn't independently reproducible down to the last border, and that gap is itself worth noticing. What matters is whether the same shape of moment shows up when the same question is asked mechanically.

### 1a. Full price convergence &mdash; 2023-12-05, 19:00 UTC

[VoltPath dashboard link](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2023-12-05&dateTo=2023-12-06). Note VoltPath displays UTC; 19:00 UTC is 20:00 CET.

In [ ]:
convergence_day = fetch_flows("2023-12-05", "2023-12-06", resolution="native")
convergence_counts = hourly_border_counts(convergence_day, "Convergence")
peak_row = peak(convergence_counts)
print(f"Peak: {peak_row['n_borders']} borders converged at {peak_row['datetime']}")

fig = px.line(
    convergence_counts,
    x="datetime",
    y="n_borders",
    title="Converged borders, 2023-12-05 (UTC)",
    labels={"n_borders": "Borders converged", "datetime": "Time"},
    color_discrete_sequence=[FLOW_COLORS["Convergence"]],
)
fig.show()

convergence_summary(convergence_day).head(10)

### 1b. Peak simultaneous congestion &mdash; 2023-02-22, 13:00 UTC

Iacopo reported 47 of 67 borders congested at 14:00 CET (13:00 UTC). [VoltPath dashboard link](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2023-02-22&dateTo=2023-02-23).

In [ ]:
congestion_day = fetch_flows("2023-02-22", "2023-02-23", resolution="native")
congestion_counts = hourly_border_counts(congestion_day, "Congestion")
peak_row = peak(congestion_counts)
print(f"Peak: {peak_row['n_borders']} borders congested at {peak_row['datetime']}")

fig = px.line(
    congestion_counts,
    x="datetime",
    y="n_borders",
    title="Congested borders, 2023-02-22 (UTC)",
    labels={"n_borders": "Borders congested", "datetime": "Time"},
    color_discrete_sequence=[FLOW_COLORS["Congestion"]],
)
fig.add_vline(x="2023-02-22 13:00:00", line_dash="dash", annotation_text="Iacopo's reported peak")
fig.show()

leaderboard = flow_type_summary(congestion_day, "Congestion").head(10)
fig2 = px.bar(
    leaderboard,
    x="implied_value_eur",
    y="border",
    orientation="h",
    title="Top 10 borders by implied congestion cost, 2023-02-22",
    labels={"implied_value_eur": "Implied cost (EUR)", "border": "Border"},
    color_discrete_sequence=[FLOW_COLORS["Congestion"]],
)
fig2.update_layout(yaxis=dict(autorange="reversed"))
fig2.show()

### 1c. Peak counter-intuitive flow &mdash; 2023-07-09, 09:00 UTC

Iacopo reported 19 borders with counter-intuitive flow at 11:00 CET (09:00 UTC). [VoltPath dashboard link](https://voltpath.ai/dashboard?dateFrom=2023-07-09&dateTo=2023-07-10&viewMode=flows-prices).

In [ ]:
ci_day = fetch_flows("2023-07-09", "2023-07-10", resolution="native")
ci_counts = hourly_border_counts(ci_day, "Counter-intuitive")
peak_row = peak(ci_counts)
print(f"Peak: {peak_row['n_borders']} borders counter-intuitive at {peak_row['datetime']}")

fig = px.line(
    ci_counts,
    x="datetime",
    y="n_borders",
    title="Counter-intuitive borders, 2023-07-09 (UTC)",
    labels={"n_borders": "Borders counter-intuitive", "datetime": "Time"},
    color_discrete_sequence=[FLOW_COLORS["Counter-intuitive"]],
)
fig.add_vline(x="2023-07-09 09:00:00", line_dash="dash", annotation_text="Iacopo's reported peak")
fig.show()

leaderboard = flow_type_summary(ci_day, "Counter-intuitive").head(10)
fig2 = px.bar(
    leaderboard,
    x="implied_value_eur",
    y="border",
    orientation="h",
    title="Top 10 borders by implied welfare value, 2023-07-09",
    labels={"implied_value_eur": "Implied value (EUR)", "border": "Border"},
    color_discrete_sequence=[FLOW_COLORS["Counter-intuitive"]],
)
fig2.update_layout(yaxis=dict(autorange="reversed"))
fig2.show()

Iacopo's article walks through why one specific counter-intuitive pattern made sense despite looking wrong: at 11:00 CET on 2023-07-09, Hungary re-exported Romanian power to Austria, Croatia, Slovenia, and Slovakia at a per-border loss, because the HU-RO price spread was so large that importing from Romania and re-exporting elsewhere was still a net win overall.

`flow_type_summary` gives the leaderboard; the cell below reproduces his specific arithmetic directly from the same peak-hour data &mdash; no manual lookup, same formula (`|net_flow_mw| x |price_delta|`) applied per border.

In [ ]:
peak_hour = ci_day[ci_day["datetime"] == peak_row["datetime"]]

# The borders Iacopo's worked example covers: HU re-exporting Romanian power onward.
re_export_borders = ["AT -> HU", "HR -> HU", "HU -> SI", "HU -> SK"]
import_border = "HU -> RO"

re_exports = peak_hour[peak_hour["border"].isin(re_export_borders)].copy()
re_exports["loss_eur"] = (re_exports["net_flow_mw"].abs() * re_exports["price_delta"].abs()).round(0)
total_loss = re_exports["loss_eur"].sum()

import_leg = peak_hour[peak_hour["border"] == import_border].iloc[0]
import_gain = round(abs(import_leg["net_flow_mw"]) * abs(import_leg["price_delta"]), 0)

print(re_exports[["border", "net_flow_mw", "price_delta", "loss_eur"]].to_string(index=False))
print(f"\nTotal re-export loss: EUR {total_loss:,.0f}")
print(f"HU-RO import gain:    EUR {import_gain:,.0f}")
print(f"Net welfare effect:   EUR {import_gain - total_loss:,.0f}")

## Part 2: the same functions on real 2024&ndash;2026 structural shifts

A one-off historical reproduction only proves the platform can look backward. The same three functions, unmodified, should also show real, ongoing shifts in how the European grid is coupled. Three verified events, each with a source, each queried the same way as everything above:

- **Baltic states synchronisation, 2025-02-07/09** &mdash; Estonia, Latvia, and Lithuania desynchronised from Russia's BRELL ring and synchronised with Continental Europe. The sharpest, most discrete break of the three. ([ENTSO-E announcement](https://www.entsoe.eu/news/2025/02/09/entso-e-confirms-successful-synchronization-of-the-continental-european-electricity-system-with-the-systems-of-the-baltic-countries/))
- **Flamanville 3 EPR grid connection, 2024-12-21** &mdash; France's first new nuclear unit in 25 years (1,630 MW) connected to the grid, ramping up through 2025. A gradual shift, not a step change. ([EDF press release](https://www.edf.fr/en/the-edf-group/dedicated-sections/journalists/all-press-releases/update-on-the-flamanville-epr-the-reactor-produces-its-first-electrons-on-the-national-electricity-grid))
- **Core "Advanced Hybrid Coupling" go-live, 2026-06-10/11** &mdash; a market-coupling methodology change on Core CCR's external borders. ([JAO confirmation](https://www.jao.eu/news/confirmation-core-advanced-hybrid-coupling-go-live))

Each window is 8 days (event date &plusmn; 4 days) at hourly resolution &mdash; wide enough to see a trend, narrow enough that the API doesn't auto-downgrade past hourly.

In [ ]:
EVENTS = [
    {"name": "Baltic sync", "date": "2025-02-08", "date_from": "2025-02-04", "date_to": "2025-02-12"},
    {"name": "Flamanville 3 connection", "date": "2024-12-21", "date_from": "2024-12-17", "date_to": "2024-12-25"},
    {"name": "Core Advanced Hybrid Coupling", "date": "2026-06-10", "date_from": "2026-06-06", "date_to": "2026-06-14"},
]

for event in EVENTS:
    df = fetch_flows(event["date_from"], event["date_to"], resolution="1H")
    frames = []
    for flow_type in ("Congestion", "Counter-intuitive"):
        counts = hourly_border_counts(df, flow_type)
        counts["flow_type"] = flow_type
        frames.append(counts)
    combined = pd.concat(frames)

    fig = px.line(
        combined,
        x="datetime",
        y="n_borders",
        color="flow_type",
        title=f"{event['name']}: congested / counter-intuitive borders, {event['date_from']} to {event['date_to']}",
        labels={"n_borders": "Borders", "datetime": "Time", "flow_type": "Flow type"},
        color_discrete_map=FLOW_COLORS,
    )
    fig.add_vline(x=event["date"], line_dash="dash", annotation_text=event["name"])
    fig.show()

### Did coupling reliability actually change?

`convergence_summary` gives a time-share per border. Splitting each event's window at the event date and comparing the average `share_converged` before vs. after is a direct, mechanical answer to "did this event measurably change how often prices converged" &mdash; the same question ENTSO-E's Bidding Zone Configuration Technical Report answers by hand for past market-coupling go-lives (its 2025 edition attributes Croatia's and Austria's 2022 congestion-income swings to that year's Core FBMC go-live, pp.60-63 covering 2021-2023 &mdash; before this API's window, but the same kind of before/after question).

In [ ]:
def before_after_convergence(df: pd.DataFrame, event_date: str) -> pd.Series:
    before = df[df["datetime"] < event_date]
    after = df[df["datetime"] >= event_date]
    return pd.Series(
        {
            "share_converged_before": convergence_summary(before)["share_converged"].mean(),
            "share_converged_after": convergence_summary(after)["share_converged"].mean(),
        }
    )


for event in EVENTS:
    df = fetch_flows(event["date_from"], event["date_to"], resolution="1H")
    comparison = before_after_convergence(df, event["date"])
    print(event["name"])
    print(comparison.to_string())
    print()

## Takeaways

- The three functions above &mdash; `hourly_border_counts`, `flow_type_summary`, `convergence_summary` &mdash; answer "how much", "where", and "how often" for any date range, not just the three hours Iacopo picked by hand.
- Reproducing a published one-off analysis exactly is hard even with the same underlying data source; small differences in border count are themselves evidence for why platforms like this exist, not a bug to chase away.
- The same functions applied to 2024&ndash;2026 structural events (a grid synchronisation, a new nuclear unit, a market-coupling methodology change) turn a historical curiosity into a live monitoring tool &mdash; point it at any date range, including next week's.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).